In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from torchvision.utils import make_grid
import torch.nn.functional as F


import re
import glob
import random

from PIL import Image
from tqdm import tqdm
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold

from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel

import warnings
# filter warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USER')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_KEY')

In [ ]:
!kaggle competitions download -c deep-learning-for-computer-vision-and-nlp-2026-10

100% 997M/997M [00:52<00:00, 19.7MB/s]



In [ ]:
!unzip -q deep-learning-for-computer-vision-and-nlp-2026-10.zip -d dataset/

## EDA

In [ ]:
train_df = pd.read_csv('dataset/train.csv', encoding='latin-1')
test_df = pd.read_csv('dataset/test.csv', encoding='latin-1')

In [ ]:
train_df

,PetID,Description,AdoptionSpeed
0,d3b4f29f8,Mayleen and Flo are two lovely adorable sister...,2
1,e9dc82251,A total of 5 beautiful Tabbys available for ad...,2
2,8111f6d4a,Two-and-a-half month old girl. Very manja and ...,2
3,693a90fda,Neil is a healthy and active ~2-month-old fema...,2
4,9d08c85ef,Gray kitten available for adoption in sungai p...,2
...,...,...,...
6426,440676acb,Puppy was rescued in the monsoon drain near my...,2
6427,62550cf78,Please contact if you are interested.,1
6428,a062234cb,4/1/ An adorable domestic medium hair kitten l...,4
6429,7a54b760e,6 Jan :30: We found this poor dog (male) wande...,2


In [ ]:
train_df.duplicated().sum()

np.int64(0)

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6431 entries, 0 to 6430
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   PetID          6431 non-null   object
 1   Description    6426 non-null   object
 2   AdoptionSpeed  6431 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 150.9+ KB


In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1891 entries, 0 to 1890
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   PetID        1891 non-null   object
 1   Description  1890 non-null   object
dtypes: object(2)
memory usage: 29.7+ KB


In [ ]:
train_df['Description'] = train_df['Description'].fillna('')
test_df['Description'] = test_df['Description'].fillna('')

In [ ]:
# Рахуємо довжину та слова в описі та кількість символів
train_grouped = (
    train_df[['PetID', 'AdoptionSpeed']]
    .assign(
        char_count=train_df['Description'].str.len(),
        word_count=train_df['Description'].apply(lambda x: len(x.split()))
    )
    .groupby('AdoptionSpeed')
    .agg(
        total_pets=('PetID', 'count'),
        mean_words=('word_count', 'mean'),
        median_words=('word_count', 'median'),
        max_words=('word_count', 'max'),
        mean_chars=('char_count', 'mean'),
        no_description_pct=('char_count', lambda x: (x == 0).mean() * 100)
    )
    .reset_index()
    .assign(**{'%': lambda df: (df['total_pets'] / len(train_df)) * 100})
)

print(train_grouped.round(2).to_string(index=False))

 AdoptionSpeed  total_pets  mean_words  median_words  max_words  mean_chars  no_description_pct     %
             1        1197       68.52          49.0       1153      369.70                0.08 18.61
             2        1773       68.71          46.0        870      374.80                0.11 27.57
             3        1328       71.75          50.5        936      391.27                0.08 20.65
             4        2133       63.37          43.0        789      346.18                0.05 33.17


Помірний дисбаланс класів. Найбільша кількість тварин належить до AdoptionSpeed 4 — 2133 тварини (33,17%), тобто вони були прилаштовані найповільніше.
Середня кількість слів в описі для груп 1–3 є близькою — від 68,52 до 71,75 слова, тоді як у групі 4 вона дещо нижча — 63,37 слова.
Медіанна довжина опису також найменша у групі 4 — 43 слова, порівняно з 46–50,5 словами в інших групах.
Максимальна довжина опису значно варіюється між групами, але сама по собі не демонструє чіткої залежності від швидкості прилаштування.
Середня кількість символів повторює тенденцію до дещо коротших описів у групі 4: 346,18 символів проти 369,70–391,27 у групах 1–3.
Частка тварин без опису є невеликою в усіх групах і становить 5–11%, причому найбільша вона у групі 2 (11%).
Загалом дані можуть свідчити про певний зв’язок між довжиною опису та швидкістю прилаштування, оскільки у найповільнішій групі 4 описи в середньому коротші, хоча за цими показниками не можна встановити причинно-наслідковий зв’язок.

In [ ]:
from collections import Counter

# Підрахунок фотографій у файловій системі
img_dir = 'dataset/images/images/train'
photo_counter = Counter()

for filename in os.listdir(img_dir):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        pet_id = filename.split('-')[0] if '-' in filename else os.path.splitext(filename)[0]
        photo_counter[pet_id] += 1

photo_counts = train_df['PetID'].map(photo_counter).fillna(0).astype(int)
total_all_photos = photo_counts.sum()

stats = (
    train_df[['PetID', 'AdoptionSpeed']]
    .assign(photo_count=photo_counts)
    .groupby('AdoptionSpeed')
    .agg(
        total_pets=('PetID', 'count'),
        total_photos=('photo_count', 'sum'),
        mean_photos=('photo_count', 'mean'),
        median_photos=('photo_count', 'median'),
        std_photos=('photo_count', 'std'),
        min_photos=('photo_count', 'min'),
        max_photos=('photo_count', 'max'),
        no_photo_pct=('photo_count', lambda x: (x == 0).mean() * 100)
    )
    .reset_index()
    .assign(**{'%': lambda df: (df['total_photos'] / total_all_photos) * 100})
)

print(stats.round(2).to_string(index=False))

 AdoptionSpeed  total_pets  total_photos  mean_photos  median_photos  std_photos  min_photos  max_photos  no_photo_pct     %
             1        1197          5000         4.18            4.0        3.25           1          30           0.0 17.56
             2        1773          8395         4.73            4.0        3.99           1          30           0.0 29.49
             3        1328          7150         5.38            4.0        4.76           1          30           0.0 25.11
             4        2133          7927         3.72            3.0        3.30           1          30           0.0 27.84


Найбільша кількість тварин також припадає на AdoptionSpeed 4 — 2133 (27,84%), тоді як найменша кількість — у групі 1 (17,56%).
Середня кількість фотографій зростає від 4,18 у групі 1 до 5,38 у групі 3, після чого знижується до 3,72 у групі 4.
Медіанна кількість фотографій становить 4 у групах 1–3 та зменшується до 3 у групі 4.
Найбільше стандартне відхилення спостерігається у групі 3 (4,76), що свідчить про більшу варіативність кількості фотографій у цій групі.
Усі тварини мають щонайменше одну фотографію, тому частка записів без фото для всіх груп становить 0%.
Найповільніше прилаштовані тварини (група 4) у середньому мають найменше фотографій, тоді як у групі 3 їх кількість найбільша.
Загалом результати вказують на можливий зв’язок між кількістю фотографій і швидкістю прилаштування: більша кількість фото характерна для груп із AdoptionSpeed 2–3, тоді як у групі 4 кількість фото помітно нижча.

In [ ]:
# Фіксація відтворюваності

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Метрика Quadratic Weighted Kappa (QWK)

def confusion_matrix(rater_a, rater_b, min_rating=None, max_rating=None):
    assert len(rater_a) == len(rater_b)
    if min_rating is None:
        min_rating = min(rater_a + rater_b)
    if max_rating is None:
        max_rating = max(rater_a + rater_b)
    num_ratings = int(max_rating - min_rating + 1)
    conf_mat = [[0 for _ in range(num_ratings)] for _ in range(num_ratings)]
    for a, b in zip(rater_a, rater_b):
        conf_mat[a - min_rating][b - min_rating] += 1
    return conf_mat

def histogram(ratings, min_rating=None, max_rating=None):
    if min_rating is None:
        min_rating = min(ratings)
    if max_rating is None:
        max_rating = max(ratings)
    num_ratings = int(max_rating - min_rating + 1)
    hist_ratings = [0 for _ in range(num_ratings)]
    for r in ratings:
        hist_ratings[r - min_rating] += 1
    return hist_ratings

def quadratic_weighted_kappa(y, y_pred):
    rater_a = np.array(y, dtype=int).tolist()
    rater_b = np.array(y_pred, dtype=int).tolist()
    min_rating = min(min(rater_a), min(rater_b))
    max_rating = max(max(rater_a), max(rater_b))

    conf_mat = confusion_matrix(rater_a, rater_b, min_rating, max_rating)
    num_ratings = len(conf_mat)
    num_scored_items = float(len(rater_a))

    hist_rater_a = histogram(rater_a, min_rating, max_rating)
    hist_rater_b = histogram(rater_b, min_rating, max_rating)

    numerator = 0.0
    denominator = 0.0

    for i in range(num_ratings):
        for j in range(num_ratings):
            expected_count = (hist_rater_a[i] * hist_rater_b[j]) / num_scored_items
            d = pow(i - j, 2.0) / pow(num_ratings - 1, 2.0)
            numerator += d * conf_mat[i][j] / num_scored_items
            denominator += d * expected_count / num_scored_items

    if denominator == 0:
        return 0.0
    return 1.0 - (numerator / denominator)


# Оптимізатор порогів для відображення неперервних прогнозів у класи
class OptimizedRounder:
    def __init__(self, labels=(0, 1, 2, 3, 4)):
        self.labels = sorted(labels)
        self.coef_ = None

    def _loss(self, coef, X, y):
        # сортування порогів, щоб вони були монотонними для np.digitize
        sorted_coef = np.sort(coef)
        binned = np.digitize(X, sorted_coef)
        # Обмежуємо індекси в межах допустимих класів
        preds = np.array([self.labels[min(idx, len(self.labels) - 1)] for idx in binned])
        return -quadratic_weighted_kappa(y, preds)

    def fit(self, X, y):
        # Початкові пороги між сусідніми класами: [0.5, 1.5, 2.5, 3.5]
        init_coef = [self.labels[i] + 0.5 for i in range(len(self.labels) - 1)]
        res = minimize(self._loss, init_coef, args=(X, y), method='Powell')
        # Зберігаємо відсортовані фінальні пороги
        self.coef_ = np.sort(res.x)

    def predict(self, X):
        if self.coef_ is None:
            raise ValueError("OptimizedRounder ще не навчено. Спочатку викличте fit().")
        binned = np.digitize(X, self.coef_)
        return np.array([self.labels[min(idx, len(self.labels) - 1)] for idx in binned])

In [ ]:
# Очищення тексту опису
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return "no description"
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)  # видалення HTML-тегів
    text = re.sub(r'[^\w\s,.!?]', ' ', text)  # видалення спецсимволів
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower() if text else "no description"

train_df = pd.read_csv('dataset/train.csv', encoding='latin-1')
test_df = pd.read_csv('dataset/test.csv', encoding='latin-1')

train_df['Description'] = train_df['Description'].fillna('').apply(clean_text)
test_df['Description'] = test_df['Description'].fillna('').apply(clean_text)

# Мапінг усіх зображень для кожного PetID
def build_image_map(image_dir):
    all_files = glob.glob(os.path.join(image_dir, '*.jpg'))
    pet_image_map = {}
    for filepath in all_files:
        filename = os.path.basename(filepath)
        pet_id = filename.split('-')[0]
        pet_image_map.setdefault(pet_id, []).append(filepath)
    # Сортування для стабільного порядку фото (1, 2, ...)
    for pet_id in pet_image_map:
        pet_image_map[pet_id].sort(key=lambda p: int(re.findall(r'-(\d+)\.jpg', p)[0]) if re.findall(r'-(\d+)\.jpg', p) else 0)
    return pet_image_map

train_image_map = build_image_map('dataset/images/images/train')
test_image_map = build_image_map('dataset/images/images/test')

train_df['image_paths'] = train_df['PetID'].map(train_image_map).apply(lambda x: x if isinstance(x, list) else [])
test_df['image_paths'] = test_df['PetID'].map(test_image_map).apply(lambda x: x if isinstance(x, list) else [])

# Розбиття на train / validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['AdoptionSpeed'])):
    train_df.loc[val_idx, 'fold'] = fold

train_split = train_df[train_df['fold'] != 0].reset_index(drop=True)
val_split = train_df[train_df['fold'] == 0].reset_index(drop=True)

Розробимо базову модель ResNet50, яка навчається лише на зображеннях.

In [ ]:
img_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PetImageDataset(Dataset):
    def __init__(self, df, transform=None, max_images=4):
        self.df = df
        self.transform = transform
        self.max_images = max_images

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_paths = row['image_paths']
        tensors = []

        if len(img_paths) == 0:
            tensors.append(torch.zeros(3, 224, 224))
        else:
            selected_paths = img_paths[:self.max_images]
            for p in selected_paths:
                try:
                    img = Image.open(p).convert('RGB')
                    if self.transform:
                        img = self.transform(img)
                    tensors.append(img)
                except Exception:
                    tensors.append(torch.zeros(3, 224, 224))

        # Падінг до max_images
        num_images = len(tensors)
        while len(tensors) < self.max_images:
            tensors.append(torch.zeros(3, 224, 224))

        stacked_images = torch.stack(tensors)  # [max_images, 3, 224, 224]
        mask = torch.zeros(self.max_images, dtype=torch.float32)
        mask[:num_images] = 1.0

        target = torch.tensor(row['AdoptionSpeed'], dtype=torch.float32) if 'AdoptionSpeed' in row else torch.tensor(0.0)
        return stacked_images, mask, target

class MultiImageResNet50(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        base = models.resnet50(weights=weights)
        self.feature_dim = base.fc.in_features  # 2048
        base.fc = nn.Identity()
        self.backbone = base

        # Механізм уваги для зважування фотографій тварини
        self.attention = nn.Sequential(
            nn.Linear(self.feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.head = nn.Sequential(
            nn.BatchNorm1d(self.feature_dim),
            nn.Dropout(0.3),
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def extract_features(self, images, mask):
        # images: [B, K, 3, 224, 224], mask: [B, K]
        B, K, C, H, W = images.shape
        flat_images = images.view(B * K, C, H, W)
        feats = self.backbone(flat_images).view(B, K, self.feature_dim)  # [B, K, 2048]

        # Обчислення ваг уваги з урахуванням маски
        scores = self.attention(feats).squeeze(-1)  # [B, K]
        scores = scores.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(scores, dim=-1).unsqueeze(-1)  # [B, K, 1]

        # Зважена сума ознак всіх зображень тварини
        pooled = torch.sum(feats * attn_weights, dim=1)  # [B, 2048]
        return pooled

    def forward(self, images, mask):
        pooled = self.extract_features(images, mask)
        out = self.head(pooled).squeeze(-1)
        return out

In [ ]:
# Навчання та оцінка моделі зображень
train_img_ds = PetImageDataset(train_split, transform=img_train_transform)
val_img_ds = PetImageDataset(val_split, transform=img_val_transform)

train_img_loader = DataLoader(train_img_ds, batch_size=32, shuffle=True, num_workers=4)
val_img_loader = DataLoader(val_img_ds, batch_size=32, shuffle=False, num_workers=4)

resnet_model = MultiImageResNet50(pretrained=True).to(device)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(resnet_model.parameters(), lr=1e-4, weight_decay=1e-2)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 40.7MB/s]


In [ ]:

def train_and_eval_image_model(model, train_loader, val_loader, epochs=4):
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for images, mask, targets in tqdm(train_loader, desc=f"ResNet Epoch {epoch+1}"):
            images, mask, targets = images.to(device), mask.to(device), targets.to(device)
            optimizer.zero_grad()
            preds = model(images, mask)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # Валідація та розрахунок QWK
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for images, mask, targets in val_loader:
            images, mask = images.to(device), mask.to(device)
            preds = model(images, mask)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(targets.numpy())

    rounder = OptimizedRounder()
    rounder.fit(np.array(val_preds), np.array(val_targets))
    discrete_preds = rounder.predict(np.array(val_preds))
    qwk = quadratic_weighted_kappa(val_targets, discrete_preds)
    print(f"ResNet50 Multi-Image Validation QWK: {qwk:.4f}")
    return model, rounder, qwk

resnet_model, img_rounder, img_qwk = train_and_eval_image_model(
    resnet_model, train_img_loader, val_img_loader, epochs=4
)

ResNet Epoch 4: 100%|██████████| 161/161 [03:47<00:00,  1.41s/it]


ResNet50 Multi-Image Validation QWK: 0.4507


Розробимо базову модель ВERT, яка навчається лише на описі.

In [ ]:
TOKENIZER_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

class PetTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['Description'].tolist()
        self.targets = df['AdoptionSpeed'].tolist() if 'AdoptionSpeed' in df else [0] * len(df)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'target': torch.tensor(self.targets[idx], dtype=torch.float32)
        }

class BertRegressor(nn.Module):
    def __init__(self, pretrained_model=TOKENIZER_NAME):
        super().__init__()
        self.bert = AutoModel.from_pretrained(pretrained_model)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.bert.config.hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def extract_features(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_feat = outputs.last_hidden_state[:, 0, :]  # [B, 768]
        return cls_feat

    def forward(self, input_ids, attention_mask):
        cls_feat = self.extract_features(input_ids, attention_mask)
        return self.head(cls_feat).squeeze(-1)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Навчання та оцінка моделі тексту
train_text_ds = PetTextDataset(train_split, tokenizer)
val_text_ds = PetTextDataset(val_split, tokenizer)

train_text_loader = DataLoader(train_text_ds, batch_size=32, shuffle=True, num_workers=4)
val_text_loader = DataLoader(val_text_ds, batch_size=32, shuffle=False, num_workers=4)

bert_model = BertRegressor().to(device)
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5, weight_decay=1e-2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def train_and_eval_text_model(model, train_loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"BERT Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            targets = batch['target'].to(device)

            bert_optimizer.zero_grad()
            preds = model(input_ids, mask)
            loss = criterion(preds, targets)
            loss.backward()
            bert_optimizer.step()

    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            preds = model(input_ids, mask)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(batch['target'].numpy())

    rounder = OptimizedRounder()
    rounder.fit(np.array(val_preds), np.array(val_targets))
    discrete_preds = rounder.predict(np.array(val_preds))
    qwk = quadratic_weighted_kappa(val_targets, discrete_preds)
    print(f"BERT Text Validation QWK: {qwk:.4f}")
    return model, rounder, qwk

bert_model, text_rounder, text_qwk = train_and_eval_text_model(
    bert_model, train_text_loader, val_text_loader, epochs=3
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BERT Epoch 3: 100%|██████████| 161/161 [01:48<00:00,  1.48it/s]


BERT Text Validation QWK: 0.3274


Розробимо модель, яка навчається на зображеннях та тексті (ResNet50 + BERT)

In [ ]:
class PetMultimodalDataset(Dataset):
    def __init__(self, df, tokenizer, img_transform=None, max_images=4, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.img_transform = img_transform
        self.max_images = max_images
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Обробка зображень
        img_paths = row['image_paths']
        tensors = []
        if len(img_paths) == 0:
            tensors.append(torch.zeros(3, 224, 224))
        else:
            for p in img_paths[:self.max_images]:
                try:
                    img = Image.open(p).convert('RGB')
                    if self.img_transform:
                        img = self.img_transform(img)
                    tensors.append(img)
                except Exception:
                    tensors.append(torch.zeros(3, 224, 224))

        num_images = len(tensors)
        while len(tensors) < self.max_images:
            tensors.append(torch.zeros(3, 224, 224))

        images = torch.stack(tensors)
        img_mask = torch.zeros(self.max_images, dtype=torch.float32)
        img_mask[:num_images] = 1.0

        # Обробка тексту
        text = str(row['Description'])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )

        target = torch.tensor(row['AdoptionSpeed'], dtype=torch.float32) if 'AdoptionSpeed' in row else torch.tensor(0.0)

        return {
            'images': images,
            'img_mask': img_mask,
            'input_ids': enc['input_ids'].squeeze(0),
            'text_mask': enc['attention_mask'].squeeze(0),
            'target': target
        }

class MultimodalPetModel(nn.Module):
    def __init__(self, img_backbone, text_backbone):
        super().__init__()
        self.img_module = img_backbone
        self.text_module = text_backbone

        fusion_dim = self.img_module.feature_dim + self.text_module.bert.config.hidden_size  # 2048 + 768 = 2816

        self.fusion_head = nn.Sequential(
            nn.BatchNorm1d(fusion_dim),
            nn.Dropout(0.4),
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, images, img_mask, input_ids, text_mask):
        img_feats = self.img_module.extract_features(images, img_mask)
        text_feats = self.text_module.extract_features(input_ids, text_mask)
        fused = torch.cat([img_feats, text_feats], dim=1)
        return self.fusion_head(fused).squeeze(-1)

In [ ]:
train_multi_ds = PetMultimodalDataset(train_split, tokenizer, img_transform=img_train_transform)
val_multi_ds = PetMultimodalDataset(val_split, tokenizer, img_transform=img_val_transform)

train_multi_loader = DataLoader(train_multi_ds, batch_size=24, shuffle=True, num_workers=4)
val_multi_loader = DataLoader(val_multi_ds, batch_size=24, shuffle=False, num_workers=4)

# Ініціалізація вагами з попередньо навчених одномодальних моделей
multi_model = MultimodalPetModel(resnet_model, bert_model).to(device)

multi_optimizer = torch.optim.AdamW([
    {'params': multi_model.img_module.parameters(), 'lr': 2e-5},
    {'params': multi_model.text_module.parameters(), 'lr': 1e-5},
    {'params': multi_model.fusion_head.parameters(), 'lr': 3e-4}
], weight_decay=1e-2)

In [ ]:
def train_and_eval_multimodal(model, train_loader, val_loader, epochs=4):
    for epoch in range(epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"Multimodal Epoch {epoch+1}"):
            images = batch['images'].to(device)
            img_mask = batch['img_mask'].to(device)
            input_ids = batch['input_ids'].to(device)
            text_mask = batch['text_mask'].to(device)
            targets = batch['target'].to(device)

            multi_optimizer.zero_grad()
            preds = model(images, img_mask, input_ids, text_mask)
            loss = criterion(preds, targets)
            loss.backward()
            multi_optimizer.step()

    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            images = batch['images'].to(device)
            img_mask = batch['img_mask'].to(device)
            input_ids = batch['input_ids'].to(device)
            text_mask = batch['text_mask'].to(device)
            preds = model(images, img_mask, input_ids, text_mask)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(batch['target'].numpy())

    rounder = OptimizedRounder()
    rounder.fit(np.array(val_preds), np.array(val_targets))
    discrete_preds = rounder.predict(np.array(val_preds))
    qwk = quadratic_weighted_kappa(val_targets, discrete_preds)
    print(f"Multimodal (ResNet50 + BERT) Validation QWK: {qwk:.4f}")
    return model, rounder, qwk

multi_model, multi_rounder, multi_qwk = train_and_eval_multimodal(
    multi_model, train_multi_loader, val_multi_loader, epochs=4
)

Multimodal Epoch 1:   0%|          | 1/215 [00:04<16:19,  4.57s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 36.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 31.81 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 13.92 GiB is allocated by PyTorch, and 478.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
test_ds = PetMultimodalDataset(test_df, tokenizer, img_transform=img_val_transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4)

multi_model.eval()
test_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        images = batch['images'].to(device)
        img_mask = batch['img_mask'].to(device)
        input_ids = batch['input_ids'].to(device)
        text_mask = batch['text_mask'].to(device)
        preds = multi_model(images, img_mask, input_ids, text_mask)
        test_preds.extend(preds.cpu().numpy())

# Дискретизація неперервних прогнозів за допомогою підібраних порогів
final_predictions = multi_rounder.predict(np.array(test_preds))

submission = pd.DataFrame({
    'PetID': test_df['PetID'],
    'AdoptionSpeed': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успішно згенеровано.")
print(submission.head())

Inference: 100%|██████████| 60/60 [00:43<00:00,  1.37it/s]

Файл submission.csv успішно згенеровано.
       PetID  AdoptionSpeed
0  6697a7f62              4
1  23b64fe21              3
2  41e824cbe              4
3  6c3d7237b              4
4  97b0b5d92              3


In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.6 MB/s eta 0:00:00


In [ ]:
# Розрахунок ваг класів на тренувальній вибірці
def get_class_weights(targets, alpha=0.5):
    classes, counts = np.unique(targets, return_counts=True)
    total_samples = len(targets)
    num_classes = len(classes)

    # Обернена частота зі згладжуванням (power smoothing)
    weights = (total_samples / (num_classes * counts)) ** alpha
    weight_dict = {cls: w for cls, w in zip(classes, weights)}
    return weight_dict

class WeightedSmoothL1Loss(nn.Module):
    def __init__(self, weight_dict, beta=1.0):
        super().__init__()
        self.beta = beta
        # Тензор ваг для класів 0, 1, 2, 3, 4
        weights_array = [weight_dict.get(i, 1.0) for i in range(5)]
        self.register_buffer('weights', torch.tensor(weights_array, dtype=torch.float32))

    def forward(self, preds, targets):
        # targets: [B], preds: [B]
        diff = torch.abs(preds - targets)
        smooth_l1 = torch.where(
            diff < self.beta,
            0.5 * (diff ** 2) / self.beta,
            diff - 0.5 * self.beta
        )

        # Визначаємо вагу кожного прикладу за його істинним цільовим класом
        int_targets = torch.clamp(targets.long(), 0, 4)
        sample_weights = self.weights[int_targets]

        weighted_loss = smooth_l1 * sample_weights
        return weighted_loss.mean()

In [ ]:
class TunableMultimodalPetModel(nn.Module):
    def __init__(self, img_module, text_module, hidden_dim_1=512, hidden_dim_2=128, dropout_1=0.4, dropout_2=0.3):
        super().__init__()
        self.img_module = img_module
        self.text_module = text_module

        fusion_dim = 2048 + 768  # 2816

        self.fusion_head = nn.Sequential(
            nn.BatchNorm1d(fusion_dim),
            nn.Dropout(dropout_1),
            nn.Linear(fusion_dim, hidden_dim_1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim_1),
            nn.Dropout(dropout_2),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Linear(hidden_dim_2, 1)
        )

    def forward(self, images, img_mask, input_ids, text_mask):
        img_feats = self.img_module.extract_features(images, img_mask)
        text_feats = self.text_module.extract_features(input_ids, text_mask)
        fused = torch.cat([img_feats, text_feats], dim=1)
        return self.fusion_head(fused).squeeze(-1)

In [ ]:
import optuna
import copy

# Приглушуємо зайвий вивід логів Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # 1. Простір гіперпараметрів для пошуку
    alpha = trial.suggest_float('alpha', 0.2, 0.7, step=0.1)
    lr_backbones = trial.suggest_float('lr_backbones', 5e-6, 3e-5, log=True)
    lr_head = trial.suggest_float('lr_head', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-4, 5e-2, log=True)

    hidden_dim_1 = trial.suggest_categorical('hidden_dim_1', [256, 512, 768])
    hidden_dim_2 = trial.suggest_categorical('hidden_dim_2', [64, 128, 256])
    dropout_1 = trial.suggest_float('dropout_1', 0.2, 0.5, step=0.1)
    dropout_2 = trial.suggest_float('dropout_2', 0.1, 0.4, step=0.1)

    # 2. Розрахунок ваг класів з поточним alpha
    train_targets = train_split['AdoptionSpeed'].values
    weight_dict = get_class_weights(train_targets, alpha=alpha)
    criterion = WeightedSmoothL1Loss(weight_dict).to(device)

    # 3. Ініціалізація моделі
    # Клонуємо базові модулі, щоб не псувати початкові ваги між спробами trial
    img_module_copy = copy.deepcopy(resnet_model)
    text_module_copy = copy.deepcopy(bert_model)

    model = TunableMultimodalPetModel(
        img_module=img_module_copy,
        text_module=text_module_copy,
        hidden_dim_1=hidden_dim_1,
        hidden_dim_2=hidden_dim_2,
        dropout_1=dropout_1,
        dropout_2=dropout_2
    ).to(device)

    optimizer = torch.optim.AdamW([
        {'params': model.img_module.parameters(), 'lr': lr_backbones},
        {'params': model.text_module.parameters(), 'lr': lr_backbones},
        {'params': model.fusion_head.parameters(), 'lr': lr_head}
    ], weight_decay=weight_decay)

    # 4. Швидке тренування (2 епохи на кожен trial для економії часу)
    TRIAL_EPOCHS = 2
    for epoch in range(TRIAL_EPOCHS):
        model.train()
        for batch in train_multi_loader:
            images = batch['images'].to(device)
            img_mask = batch['img_mask'].to(device)
            input_ids = batch['input_ids'].to(device)
            text_mask = batch['text_mask'].to(device)
            targets = batch['target'].to(device)

            optimizer.zero_grad()
            preds = model(images, img_mask, input_ids, text_mask)
            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

    # 5. Оцінка валідаційного QWK
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_multi_loader:
            images = batch['images'].to(device)
            img_mask = batch['img_mask'].to(device)
            input_ids = batch['input_ids'].to(device)
            text_mask = batch['text_mask'].to(device)

            preds = model(images, img_mask, input_ids, text_mask)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(batch['target'].numpy())

    val_preds = np.array(val_preds)
    val_targets = np.array(val_targets)

    # Перевірка на вироджені передбачення
    if np.std(val_preds) < 0.05:
        return 0.0

    rounder = OptimizedRounder()
    rounder.fit(val_preds, val_targets)
    discrete_preds = rounder.predict(val_preds)
    qwk = quadratic_weighted_kappa(val_targets, discrete_preds)

    return qwk

# Запуск пошуку (наприклад, 10 ітерацій)
print("--- Запуск оптимізації Optuna ---")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=10, timeout=3600)

print("\nНайкращий результат знайдено!")
print(f"Кращий валідаційний QWK: {study.best_value:.4f}")
print("Оптимальні гіперпараметри:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

--- Запуск оптимізації Optuna ---

Найкращий результат знайдено!
Кращий валідаційний QWK: 0.5192
Оптимальні гіперпараметри:
  alpha: 0.4
  lr_backbones: 2.7464363002855083e-05
  lr_head: 0.0005395030966670229
  weight_decay: 0.004128205343826225
  hidden_dim_1: 256
  hidden_dim_2: 64
  dropout_1: 0.2
  dropout_2: 0.4


In [ ]:
# Оптимальні гіперпараметри:
#   alpha: 0.4
#   lr_backbones: 2.7464363002855083e-05
#   lr_head: 0.0005395030966670229
#   weight_decay: 0.004128205343826225
#   hidden_dim_1: 256
#   hidden_dim_2: 64
#   dropout_1: 0.2
#   dropout_2: 0.4


best = study.best_params

# 1. Зважування з оптимізованим alpha
train_targets = train_split['AdoptionSpeed'].values
best_weights = get_class_weights(train_targets, alpha=best['alpha'])
final_criterion = WeightedSmoothL1Loss(best_weights).to(device)

# 2. Модель з оптимальною архітектурою
final_multi_model = TunableMultimodalPetModel(
    img_module=resnet_model,
    text_module=bert_model,
    hidden_dim_1=best['hidden_dim_1'],
    hidden_dim_2=best['hidden_dim_2'],
    dropout_1=best['dropout_1'],
    dropout_2=best['dropout_2']
).to(device)

final_optimizer = torch.optim.AdamW([
    {'params': final_multi_model.img_module.parameters(), 'lr': best['lr_backbones']},
    {'params': final_multi_model.text_module.parameters(), 'lr': best['lr_backbones']},
    {'params': final_multi_model.fusion_head.parameters(), 'lr': best['lr_head']}
], weight_decay=best['weight_decay'])

# 3. Фінальне навчання
EPOCHS = 4
for epoch in range(EPOCHS):
    final_multi_model.train()
    total_loss = 0.0
    for batch in tqdm(train_multi_loader, desc=f"Final Epoch {epoch+1}/{EPOCHS}"):
        images = batch['images'].to(device)
        img_mask = batch['img_mask'].to(device)
        input_ids = batch['input_ids'].to(device)
        text_mask = batch['text_mask'].to(device)
        targets = batch['target'].to(device)

        final_optimizer.zero_grad()
        preds = final_multi_model(images, img_mask, input_ids, text_mask)
        loss = final_criterion(preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_multi_model.parameters(), max_norm=1.0)
        final_optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_multi_loader):.4f}")

# 4. Валідація та підбір порогів
final_multi_model.eval()
val_preds, val_targets = [], []
with torch.no_grad():
    for batch in val_multi_loader:
        images = batch['images'].to(device)
        img_mask = batch['img_mask'].to(device)
        input_ids = batch['input_ids'].to(device)
        text_mask = batch['text_mask'].to(device)
        preds = final_multi_model(images, img_mask, input_ids, text_mask)
        val_preds.extend(preds.cpu().numpy())
        val_targets.extend(batch['target'].numpy())

opt_rounder = OptimizedRounder()
opt_rounder.fit(np.array(val_preds), np.array(val_targets))
final_discrete_preds = opt_rounder.predict(np.array(val_preds))
final_qwk = quadratic_weighted_kappa(val_targets, final_discrete_preds)

print(f"QWK після Optuna та зважування класів: {final_qwk:.4f}")


Final Epoch 1/4: 100%|██████████| 215/215 [05:32<00:00,  1.55s/it]


Epoch 1 Loss: 0.5155


Final Epoch 2/4: 100%|██████████| 215/215 [05:32<00:00,  1.55s/it]


Epoch 2 Loss: 0.2739


Final Epoch 3/4: 100%|██████████| 215/215 [05:31<00:00,  1.54s/it]


Epoch 3 Loss: 0.2163


Final Epoch 4/4: 100%|██████████| 215/215 [05:32<00:00,  1.55s/it]

Epoch 4 Loss: 0.1675


QWK після Optuna та зважування класів: 0.4979


### Результати моделей :

**ResNet-50 (Multi-Image + Attention)**  
QWK: **0.4507**

**BERT (bert-base-uncased)**  
QWK: **0.3274**

**Late Fusion (ResNet50 + BERT)**  
QWK: **0.5038**

**Late Fusion (ResNet50 + BERT) Optuna + Class Weighting**  
QWK: **0.4979**

Зниження QWK з 0.5038 до 0.4979 показує, що додавання зважування класів у цій задачі не покращило модель.

Class Weighting змушує модель сильніше враховувати рідкісні класи, але через це вона може робити більше помилок між віддаленими класами, що сильно погіршує QWK.
OptimizedRounder уже підбирає оптимальні пороги саме під QWK, тому додаткове зважування може конфліктувати з цим підходом.
Optuna навчала моделі лише 2 епохи під час пошуку, тому знайдені параметри могли бути оптимальними для короткого навчання, але не для повноцінного тренування.

Отже, для цієї задачі базова мультимодальна модель ResNet50 + BERT дала кращий результат — QWK 0.5038.